# Minicurso de Introdução à Modelagem de Risco de Crédito — Dia 1

Este notebook acompanha o **Dia 1** do minicurso e foi pensado para ser usado **após a apresentação dos slides**.

## Objetivos do notebook
- carregar e entender a base de dados;
- definir a variável alvo de inadimplência;
- identificar desbalanceamento;
- dividir a base em treino, validação e teste;
- construir uma transformação simples de variável contínua em faixas.

## Observação
A ideia aqui não é ajustar modelos ainda. O foco do Dia 1 é **entender o problema, os dados e a preparação da base**.


## 1. Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

## 2. Carregando a base de dados

Usaremos a base `credit-g`, disponível no OpenML. Ela é uma base clássica em ensino de risco de crédito.

In [ ]:
credit = fetch_openml(name="credit-g", version=1, as_frame=True)

X = credit.data.copy()
y = credit.target.copy()

dados = X.copy()
dados["class"] = y

dados.head()

## 3. Primeira inspeção da base

Aqui queremos responder perguntas simples:

- quantas linhas e colunas temos?
- quais variáveis existem?
- quais parecem numéricas ou categóricas?

In [ ]:
print("Dimensão da base:", dados.shape)
print("\nColunas:")
print(dados.columns.tolist())

In [ ]:
dados.dtypes

## 4. Olhando a base de forma mais concreta

Além do `head()`, vale observar algumas linhas aleatórias para perceber a variedade dos perfis.

In [ ]:
dados.sample(5, random_state=42)

## 5. Explorando algumas variáveis

Antes de pensar em modelo, precisamos entender o que existe na base.

In [ ]:
dados[["duration", "credit_amount", "age"]].describe()

In [ ]:
dados["purpose"].value_counts()

In [ ]:
dados["housing"].value_counts()

### Atividade de leitura 1

Sem fazer conta ainda, observe as saídas acima e responda mentalmente:

- quais variáveis parecem numéricas?
- quais parecem categóricas?
- quais podem carregar informação sobre risco?


## 6. Definindo a variável alvo

A variável original `class` assume os valores `good` e `bad`. Vamos transformá-la em uma variável binária:

- `0` → adimplente
- `1` → inadimplente

In [ ]:
dados["class"].value_counts()

In [ ]:
dados["target"] = (dados["class"] == "bad").astype(int)

dados[["class", "target"]].head(10)

## 7. Distribuição da variável alvo

Agora queremos ver quantos clientes estão em cada classe.

In [ ]:
dados["target"].value_counts()

In [ ]:
dados["target"].value_counts(normalize=True)

In [ ]:
freq = dados["target"].value_counts().sort_index()
freq.index = ["Adimplente (0)", "Inadimplente (1)"]

plt.figure(figsize=(7, 4))
plt.bar(freq.index, freq.values)
plt.title("Distribuição da variável alvo")
plt.ylabel("Frequência")
plt.show()

### Atividade de leitura 2

Observe a distribuição da variável alvo e responda:

1. A base parece balanceada?
2. Qual classe parece mais rara?
3. Em risco de crédito, qual classe tende a ser mais importante identificar?


## 8. Por que isso importa?

Quando a classe de interesse é rara, métricas simples como acurácia podem enganar.

Um modelo pode parecer bom apenas por prever quase sempre a classe majoritária.

### Questão para reflexão

Se a maioria dos clientes é adimplente, o que acontece se um modelo prever sempre **"adimplente"**?

- Ele teria alta acurácia?
- Ele seria útil para risco de crédito?


## 9. Divisão da base em treino, validação e teste

Vamos dividir a base em duas etapas:

1. 80% treino inicial + 20% teste
2. dos 80% de treino inicial, separar 75% para treino final e 25% para validação

Resultado final esperado:
- treino: 60%
- validação: 20%
- teste: 20%

In [ ]:
# 1ª divisão: 80% treino inicial e 20% teste
X_train, X_test, y_train, y_test = train_test_split(
    dados.drop(columns=["class", "target"]),
    dados["target"],
    test_size=0.2,
    random_state=42,
    stratify=dados["target"]
)

# 2ª divisão: dos 80% iniciais, 75% ficam em treino final e 25% em validação
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

print("Treino:", len(X_train))
print("Validação:", len(X_valid))
print("Teste:", len(X_test))

## 10. Verificando a estratificação

Como usamos `stratify`, esperamos que a proporção de inadimplentes fique parecida nos três conjuntos.

In [ ]:
print("Proporção de inadimplentes no treino:    ", round(y_train.mean(), 4))
print("Proporção de inadimplentes na validação:", round(y_valid.mean(), 4))
print("Proporção de inadimplentes no teste:    ", round(y_test.mean(), 4))

### Atividade de leitura 3

Compare as três proporções acima.

1. Elas ficaram parecidas?
2. O que poderia acontecer se a divisão fosse feita sem estratificação?


## 11. Exemplo de transformação: agrupando idade

Variáveis contínuas podem ser agrupadas em faixas para:

- facilitar interpretação;
- resumir padrões;
- reduzir sensibilidade a valores extremos.

Isso é comum em aplicações de crédito, especialmente em análises exploratórias e scorecards tradicionais.

In [ ]:
dados["age"].describe()

A idade varia de 19 a 75 anos, com maior concentração entre aproximadamente 27 e 42 anos.

In [ ]:
dados["faixa_idade"] = pd.cut(
    dados["age"],
    bins=[0, 25, 35, 50, 100],
    labels=["até 25", "26–35", "36–50", "50+"]
)

dados[["age", "faixa_idade"]].head(10)

In [ ]:
faixas = dados["faixa_idade"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(faixas.index.astype(str), faixas.values)
plt.title("Distribuição das faixas de idade")
plt.ylabel("Frequência")
plt.show()

Agrupar a idade em faixas reduz a variabilidade dos dados e facilita a interpretação dos padrões de risco.

## 12. Cruzando faixa etária e inadimplência

Agora vamos olhar, de forma exploratória, como a inadimplência se distribui entre as faixas de idade.


In [ ]:
tab_idade = pd.crosstab(dados["faixa_idade"], dados["target"], normalize="index")
tab_idade.columns = ["Adimplente", "Inadimplente"]
tab_idade

### Atividade de leitura 4

Observe a tabela acima.

1. Alguma faixa parece ter proporção maior de inadimplência?
2. Isso significa causalidade?
3. Que outras variáveis poderiam interferir nessa relação?


## 13. Tratamento pré-modelagem: checagens básicas

Antes de modelar, precisamos verificar se a base está coerente. Nesta etapa, vamos observar:

- dados faltantes
- duplicatas
- tipos das variáveis
- valores potencialmente impossíveis

Mesmo quando a base está "limpa", fazer esse checklist é uma boa prática.


In [ ]:
# Dados faltantes por coluna
faltantes = dados.isna().sum().sort_values(ascending=False)
faltantes.head(10)


In [ ]:
# Número de linhas duplicadas
dados.duplicated().sum()


In [ ]:
# Tipos das variáveis
dados.dtypes.value_counts()


In [ ]:
# Checagem simples de valores potencialmente problemáticos
print("Idades < 18:", (dados["age"] < 18).sum())
print("Duração <= 0:", (dados["duration"] <= 0).sum())
print("Crédito <= 0:", (dados["credit_amount"] <= 0).sum())


### Atividade de leitura 5

Observe as checagens acima e responda:

1. A base possui muitos dados faltantes?
2. Encontramos duplicatas?
3. Algum valor parece claramente impossível?

> Em uma base real, essas verificações ajudam a evitar erros de modelagem.


## 14. Transformações simples de variáveis

Além de limpar os dados, podemos criar variáveis mais informativas.

Exemplos:
- usar o log do valor do crédito
- criar uma medida de crédito por mês
- comparar discretizações diferentes da mesma variável

Essas transformações podem tornar os padrões de risco mais visíveis.


In [ ]:
import numpy as np

# Transformações simples
dados["log_credit_amount"] = np.log1p(dados["credit_amount"])
dados["credit_per_month"] = dados["credit_amount"] / dados["duration"]

dados[["credit_amount", "log_credit_amount", "duration", "credit_per_month"]].head()


A variável `log_credit_amount` reduz a assimetria do valor do crédito, enquanto `credit_per_month` combina valor e prazo em uma medida simples de esforço mensal.


## 15. Comparando duas discretizações manuais da idade

A discretização que usamos antes foi definida manualmente. Mas será que outra divisão seria melhor para separar clientes bons e maus?

Vamos comparar duas possibilidades.


In [ ]:
# Divisão A
dados["faixa_A"] = pd.cut(
    dados["age"],
    bins=[0, 30, 45, 60, 100],
    labels=["até 30", "31–45", "46–60", "60+"]
)

# Divisão B
dados["faixa_B"] = pd.cut(
    dados["age"],
    bins=[0, 25, 35, 50, 100],
    labels=["até 25", "26–35", "36–50", "50+"]
)

tabela_A = pd.crosstab(dados["faixa_A"], dados["target"], normalize="index")
tabela_B = pd.crosstab(dados["faixa_B"], dados["target"], normalize="index")

print("=== Divisão A ===")
display(tabela_A)

print("=== Divisão B ===")
display(tabela_B)


In [ ]:
# Comparando a diferença entre a maior e a menor proporção de inadimplentes
dif_A = tabela_A[1].max() - tabela_A[1].min()
dif_B = tabela_B[1].max() - tabela_B[1].min()

print("Diferença de risco - Divisão A:", round(dif_A, 4))
print("Diferença de risco - Divisão B:", round(dif_B, 4))


### Atividade de leitura 6

Compare as duas divisões:

1. Em qual delas a proporção de inadimplentes varia mais entre as faixas?
2. Qual parece separar melhor clientes de maior e menor risco?
3. Isso sugere que uma discretização "boa" depende apenas da intuição?


## 16. WoE e IV: ideias importantes em modelagem de crédito

Em credit scoring, uma técnica clássica é transformar faixas de variáveis em números usando o **WoE (Weight of Evidence)**.

Para a faixa $j$:

$$
WoE_j = \ln\left(\frac{\%\,bons_j}{\%\,maus_j}\right)
$$

Também podemos medir a força preditiva da variável usando o **IV (Information Value)**:

$$
IV = \sum_j (\%\,bons_j - \%\,maus_j) \cdot WoE_j
$$

A ideia central é simples: uma boa divisão cria grupos com comportamentos diferentes de risco.


In [ ]:
# Instalação do pacote para WoE/IV no Colab, se necessário
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "scorecardpy", "-q"])


In [ ]:
import scorecardpy as sc


In [ ]:
def calcular_woe(df, var_faixa, target):
    tab = pd.crosstab(df[var_faixa], df[target])

    if 0 not in tab.columns:
        tab[0] = 0
    if 1 not in tab.columns:
        tab[1] = 0

    tab = tab[[0, 1]]
    tab.columns = ["bons", "maus"]

    total_bons = tab["bons"].sum()
    total_maus = tab["maus"].sum()

    tab["prop_bons"] = tab["bons"] / total_bons
    tab["prop_maus"] = tab["maus"] / total_maus

    eps = 1e-6
    tab["woe"] = np.log((tab["prop_bons"] + eps) / (tab["prop_maus"] + eps))
    tab["iv_parcial"] = (tab["prop_bons"] - tab["prop_maus"]) * tab["woe"]

    return tab


In [ ]:
woe_A = calcular_woe(dados, "faixa_A", "target")
woe_B = calcular_woe(dados, "faixa_B", "target")

print("=== WoE da Divisão A ===")
display(woe_A)

print("IV total - Divisão A:", round(woe_A["iv_parcial"].sum(), 6))

print("\n=== WoE da Divisão B ===")
display(woe_B)

print("IV total - Divisão B:", round(woe_B["iv_parcial"].sum(), 6))


Observe que:

- WoE positivo sugere maior presença relativa de bons clientes
- WoE negativo sugere maior presença relativa de maus clientes
- IV maior indica, em geral, maior capacidade de separação

Assim, podemos comparar discretizações manuais antes mesmo de ajustar um modelo.


## 17. Binning automático com `scorecardpy`

Agora vamos deixar que um método automático proponha faixas com base nos dados.

A ideia é comparar:

- discretização manual
- discretização orientada por dados

Isso ajuda a mostrar por que o tratamento pré-modelagem é tão importante em crédito.


In [ ]:
base_woe = dados.copy()
base_woe["inad"] = base_woe["target"]

vars_exemplo = [
    "age",
    "credit_amount",
    "duration",
    "credit_per_month",
    "log_credit_amount",
    "inad"
]

base_woe = base_woe[vars_exemplo]

bins = sc.woebin(base_woe, y="inad")


In [ ]:
# Tabela de bins para a idade
bins["age"]


In [ ]:
# Tabela de Information Value
iv = sc.iv(base_woe, y="inad")
iv.sort_values("info_value", ascending=False)


In [ ]:
# Aplicando a transformação WoE
base_woe_transf = sc.woebin_ply(base_woe, bins)
base_woe_transf.head()


### Atividade de leitura 7

Com base nos resultados de WoE e IV:

1. A discretização automática parece coerente?
2. Quais variáveis parecem mais informativas?
3. O que você ganha ao transformar uma faixa em um valor WoE?


## 18. Síntese ampliada do Dia 1

Até aqui, fizemos oito coisas importantes:

- entendemos a estrutura da base
- definimos a variável alvo
- discutimos desbalanceamento
- dividimos a base em treino, validação e teste
- criamos transformações simples
- comparamos discretizações manuais
- calculamos WoE e IV
- aplicamos binning automático em Python

> No Dia 2, vamos usar essas informações para construir e avaliar modelos.


## 19. Exercícios adicionais

### Exercício 8
Por que duas discretizações diferentes da mesma variável podem levar a níveis diferentes de separação entre bons e maus clientes?

**Resposta:**

### Exercício 9
O que significa um WoE negativo em uma faixa?

**Resposta:**

### Exercício 10
Qual é a diferença entre uma discretização manual e uma discretização orientada por dados?

**Resposta:**


## 20. Síntese do bloco inicial do Dia 1

Até aqui, fizemos cinco coisas importantes:

- entendemos a estrutura da base;
- definimos a variável alvo;
- verificamos o desbalanceamento;
- dividimos a base corretamente;
- construímos uma transformação simples de variável contínua.

## 21. Exercícios finais

Responda às questões abaixo com base no que foi visto no notebook.


### Exercício 1
Qual é a diferença entre a variável original `class` e a variável criada `target`?

**Resposta:**

### Exercício 2
Por que a variável `target` é essencial para formular o problema de risco de crédito como classificação binária?

**Resposta:**

### Exercício 3
O que significa dizer que uma base é desbalanceada?

**Resposta:**

### Exercício 4
Explique por que alta acurácia não garante um bom modelo em risco de crédito.

**Resposta:**

### Exercício 5
Qual é o papel dos conjuntos de treino, validação e teste?

**Resposta:**

### Exercício 6
Por que usamos `stratify` na divisão da base?

**Resposta:**

### Exercício 7
Qual a vantagem de transformar uma variável contínua, como idade, em faixas?

**Resposta:**